## Bonus: Evaluation and Production Patterns

This notebook is a reference guide — you don't need to run it live. It covers the production failure modes that bite every team building agentic RAG systems, and the retrieval evaluation metrics you need to know whether your system is actually working.

**Key concepts**: intent bleed, slot hallucination, retrieval evaluation, MRR, NDCG, precision@k

## Retrieval Evaluation Metrics

If you can't measure it, you can't improve it. These are the metrics that matter for retrieval quality.

In [ ]:
def reciprocal_rank(results: list[str], relevant: set[str]) -> float:
    """MRR: How high up is the first relevant result? Higher is better. Max = 1.0"""
    for i, r in enumerate(results, 1):
        if r in relevant:
            return 1.0 / i
    return 0.0


def precision_at_k(results: list[str], relevant: set[str], k: int) -> float:
    """Precision@k: What fraction of the top k results are relevant?"""
    top_k = results[:k]
    return sum(1 for r in top_k if r in relevant) / k


def ndcg_at_k(results: list[str], relevant: set[str], k: int) -> float:
    """NDCG@k: Are the best results ranked at the top? Higher is better. Max = 1.0"""
    import math
    dcg = sum(
        (1 if results[i] in relevant else 0) / math.log2(i + 2)
        for i in range(min(k, len(results)))
    )
    ideal = sum(1 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / ideal if ideal > 0 else 0.0


# Example: searching for "black dress" returned these products
results = ["Washington dress", "Colette dress", "GLASSIG ESPADRILLE", "Rihanna dress", "Atlas jacket"]
relevant = {"Washington dress", "Colette dress", "Rihanna dress"}  # ground truth

print(f"MRR:           {reciprocal_rank(results, relevant):.3f}  (first relevant result at rank 1 = perfect)")
print(f"Precision@3:   {precision_at_k(results, relevant, k=3):.3f}  (2 of top 3 are relevant)")
print(f"Precision@5:   {precision_at_k(results, relevant, k=5):.3f}  (3 of top 5 are relevant)")
print(f"NDCG@5:        {ndcg_at_k(results, relevant, k=5):.3f}  (relevant results near top = high score)")

## Production Failure Modes

### Intent Bleed
When context from a previous intent leaks into the current one. The slot filler starts suggesting values from a billing conversation during a product search.

### Slot Hallucination
When the LLM fills in slot values that the user never provided. "I see you're looking for contracts from March 2023" — when the user never mentioned a date.

### Low-Confidence Deadlocks
When the intent classifier can't decide between two intents and the system keeps asking the user to clarify without making progress.

### Over-Eager Reclassification
When the intent classifier reclassifies every follow-up message as a new intent instead of treating it as continuation of the current conversation.

In [ ]:
# ── Failure Mode 1: Intent Bleed ─────────────────────────────────────────────
# Slots from a previous intent leak into the next one.
# Fix: always call session.reset_intent() on intent switch — which our orchestrator does.

print("=== Intent Bleed ===")
print("User: 'I want to return my jacket'  → intent: support, slots: {issue: 'return', product: 'jacket'}")
print("User: 'Show me blue trousers'        → intent switches to product")
print("WITHOUT reset: slots still contain {issue: 'return'} — support context bleeds into product search")
print("WITH reset:    slots cleared on switch — clean product search")
print()

# ── Failure Mode 2: Slot Hallucination ───────────────────────────────────────
# The LLM fills in slot values the user never provided.
# Fix: strict instructions — "NEVER hallucinate slot values, only extract what the user said"

print("=== Slot Hallucination ===")
print("User: 'I'm looking for something nice'")
print("BAD:  extracted_slots = {'product_keyword': 'dress', 'colour': 'white'}  ← hallucinated")
print("GOOD: extracted_slots = {}  missing_required = ['product_keyword']       ← asks for clarification")
print()

# ── Failure Mode 3: Low-Confidence Deadlock ──────────────────────────────────
# Classifier can't decide between intents and keeps asking the user to clarify.
# Fix: CONFIDENCE_THRESHOLD + fallback response that names the options clearly.

print("=== Low-Confidence Deadlock ===")
print("User: 'Can you help me?'  → confidence: 0.3 (below threshold of 0.6)")
print("GOOD: 'I can help with product search, billing questions, or order issues. What do you need?'")
print("BAD:  keep re-asking 'What do you mean?' every turn indefinitely")
print()

# ── Failure Mode 4: Over-Eager Reclassification ──────────────────────────────
# Every follow-up message gets classified as a new intent instead of continuing.
# Fix: INTENT_SWITCH_THRESHOLD (0.7) — only switch if new intent is very confident.

print("=== Over-Eager Reclassification ===")
print("User: 'Show me black jackets'     → intent: product (0.9)")
print("User: 'What about blue ones?'     → classifier might say product (0.6) or unknown (0.5)")
print("WITHOUT threshold: resets slots — loses 'jacket' context")
print("WITH threshold:    stays in product intent, adds colour='blue' to existing slots")

## Key Takeaways

- **MRR (Mean Reciprocal Rank)** — How high up is the first relevant result?
- **NDCG (Normalized Discounted Cumulative Gain)** — Are the best results at the top?
- **Precision@k** — What fraction of the top k results are relevant?

These metrics tell you whether your retrieval pipeline is actually working — before the LLM ever sees the context.